# Gold Aggregation

- Should run dbt models
- Or write Pyspark gold aggregation

Setup and Configuration

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print("Starting Gold aggregation notebook...")
print("This notbeook triggers dbt models via subprocess calls")

Script to run DBT command

In [0]:
import subprocess
import os
import sys

def run_dbt_command(command: str) -> bool:
  """
  Runs a dbt command inside the Databricks notebook.
  Returns True if successful, False if failed.
  """
  # Set environment variables for DBT
  env = os.environ.copy()
  env["DATABRICKS_HOST"] = dbutils.secrets.get(
    scope = "ipl-secrets", key = "databricks-host"
  ) if "dbutils" in dir() else os.getenv("DATABRICKS_HOST")
  env["DATABRICKS_TOKEN"] = dbutils.secrets.get(
    scope = "ipl-secrets", key = "databricks-token"
  ) if "dbutils" in dir() else os.getenv("DATABRICKS_TOKEN")
  env["DATABRICKS_HTTP_PATH"] = dbutils.secrets.get(
    scope = "ipl-secrets", key = "databricks-http-path"
  ) if "dbutils" in dir() else os.getenv("DATABRICKS_HTTP_PATH")
  
  print(f"Running dbt {command}")

  result = subprocess.run(
    f"dbt {command}",
    shell = True,
    capture_output = True,
    text = True,
    env = env,
    cwd = "/Workspace/ipl-analytics-platform/dbt"
  )

  print(result.stdout)

  if result.returncode != 0 :
    print(f"ERROR: {result.stderr}")
    return False
  
  return True

Run Staging models first

In [0]:
staging_success = run_dbt_command("run --select staging")

if not staging_success :
  raise Exception (
      "dbt staging models failed."
      "Check silver tables exist before running Gold."
    )
  
print("Staging models complete ✓")

Run Gold models

In [0]:
gold_success = run_dbt_command("run --select gold")

if not gold_success :
  raise Exception (
      "dbt Gold models failed."
      "Check staging models ran successfully."
    )
  
print("Gold models complete ✓")

Run dbt tests

In [0]:
test_success = run_dbt_command("test --select gold")

if not test_success :
  raise Exception (
      "dbt tests failed on Gold models."
      "Check data quality before serving to dashboard."
    )
print("dbt tests passed ✓")

Verify gold tables created

In [0]:
gold_tables = [
  "ipl_catalog.gold.player_season_stats",
  "ipl_catalog.gold.team_win_rates",
  "ipl_catalog.gold.venue_avg_scores",
  "ipl_catalog.gold.head_to_head",
  "ipl_catalog.gold.bowler_economy",
  "ipl_catalog.gold.match_summary"
]

print("\nGold table row counts:")
for table in gold_tables:
  try :
    count = spark.table(table).count()
    print(f"{table:50} : {count:,} rows")
  except Exception as e :
    print(f"{table:50} : ERROR - {e}")

print("\n03_gold_aggregation complete ✓")